# AIC25 — Fast Tier-1 Path (Single-Camera JSONs + Tracklet Repair)

**Goal:** generate single-camera tracking JSONs for a few Warehouse_016 cameras **without training the detector** (ByteTrack fallback), then run Hakan's `tracklet_repair` ablation to produce the **Tier-1 comparison tables** — all in one run.

- Branch: **`hithesh/combined-pipeline`** (must be pushed first: `git push -u origin hithesh/combined-pipeline`).
- **No [A] training step** — skipped on purpose. Detection uses the `bytetrack_x_mot17` fallback.
- Pick a small `CAMERAS` subset in Step 4 to keep it fast.
- Run cells top to bottom on a **GPU** runtime (Runtime → Change runtime type → T4 GPU).

---
## Step 0 — Environment + Drive

In [ ]:
import os, sys

ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB:
    REPO  = '/content/repo'
    PY    = 'python'
    DRIVE = '/content/drive/MyDrive/AIC25'
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    else:
        print('Drive already mounted.')
    for d in ['models', 'outputs/Detection', 'outputs/EmbedFeature', 'outputs/Tracking']:
        os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    print(f'Environment : Colab  |  Drive : {DRIVE}')
else:
    REPO  = '/home/seco/deepLearning/Single-Camera-Tracking-Consistency'
    PY    = f'{REPO}/.venv/bin/python'
    DRIVE = None
    os.chdir(REPO)
    print('Environment : Local')
print(f'REPO : {REPO}')

---
## Step 1 — Clone repo, checkout combined branch, install deps *(Colab only)*

In [ ]:
if ON_COLAB:
    import subprocess as _sp
    if not os.path.exists(REPO):
        os.system(f'git clone https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git {REPO}')
        print('Repo cloned.')
    else:
        os.system(f'git -C {REPO} fetch --quiet')
    os.chdir(REPO)

    # Checkout the COMBINED branch (pipeline + tracklet_repair). Must be pushed to origin.
    rc = os.system(f'git -C {REPO} checkout hithesh/combined-pipeline')
    os.system(f'git -C {REPO} pull --quiet 2>/dev/null')
    if rc != 0 or not os.path.isdir(f'{REPO}/tracklet_repair'):
        raise RuntimeError(
            'Could not checkout hithesh/combined-pipeline OR tracklet_repair/ is missing.\n'
            'Push the branch first on your machine:\n'
            '    git push -u origin hithesh/combined-pipeline')
    print('On branch hithesh/combined-pipeline  |  tracklet_repair/ present ✓')

    print('Installing dependencies...')
    _pkgs = ['thop','loguru','lap','motmetrics','filterpy','easydict','yacs',
             'termcolor','prettytable','tabulate','ninja','cython_bbox','pycocotools']
    for _p in _pkgs:
        _r = _sp.run(['pip','install','-q',_p], capture_output=True, text=True)
        if _r.returncode != 0:
            print(f'  {_p}: FAILED — {_r.stderr.strip()[-80:]}')
    if _sp.run(['pip','install','-q','faiss-gpu'], capture_output=True).returncode != 0:
        _sp.run(['pip','install','-q','faiss-cpu'], capture_output=True)
    os.chdir(f'{REPO}/BoT-SORT');         os.system('python setup.py develop --quiet 2>/dev/null')
    os.chdir(f'{REPO}/deep-person-reid'); os.system('python setup.py develop --quiet 2>/dev/null')
    os.chdir(REPO)
    os.system('pip install -q -r tracking/requirements.txt 2>/dev/null')
    _v = _sp.run('python -c "from yolox.exp import get_exp; from tracker.bot_sort import BoTSORT; print(\'ALL OK\')"',
                 shell=True, capture_output=True, text=True)
    print('✓ Dependencies verified.' if 'ALL OK' in _v.stdout else '✗ FAILED: '+_v.stderr.strip()[-400:])
else:
    print('Local: using .venv — skipping.')

---
## Step 2 — Check GPU (required)

In [ ]:
import subprocess
r = subprocess.run([PY,'-c',
    'import torch; print("CUDA:", torch.cuda.is_available()); '
    'print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")'],
    capture_output=True, text=True)
print(r.stdout.strip())
if ON_COLAB and 'CUDA: False' in r.stdout:
    raise RuntimeError('NO GPU — Runtime → Change runtime type → T4 GPU, then Restart, then re-run from Step 0.')

---
## Step 3 — Download models (OSNet + ByteTrack fallback)
No AIC25 detector needed — fallback is used automatically.

In [ ]:
if ON_COLAB:
    os.system('pip install -q gdown')
    M = f'{DRIVE}/models'
    def get_model(local, on_drive, gid, name):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        if os.path.exists(local):
            print(f'{name}: already local'); return
        if os.path.exists(on_drive):
            os.system(f'cp "{on_drive}" "{local}"'); print(f'{name}: copied from Drive'); return
        print(f'{name}: downloading...')
        for attempt in range(3):                      # OSNet gdown is flaky — retry
            os.system(f'gdown "https://drive.google.com/uc?id={gid}" -O "{on_drive}"')
            if os.path.exists(on_drive) and os.path.getsize(on_drive) > 100000:
                break
            print(f'  retry {attempt+1}/3...')
        if os.path.exists(on_drive):
            os.system(f'cp "{on_drive}" "{local}"'); print(f'{name}: done (cached to Drive)')
        else:
            print(f'{name}: STILL MISSING — download manually from the torchreid model zoo and place at {local}')
    get_model(f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar',
              f'{M}/osnet_ms_m_c.pth.tar', '1IosIFlLiulGIjwW3H8uMCC3YvMyr9gZ2', 'OSNet')
    get_model(f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar',
              f'{M}/bytetrack_x_mot17.pth.tar', '1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5', 'ByteTrack')
    print('AIC25 detector: NOT used in fast path — ByteTrack fallback handles detection.')
else:
    print('Local: models already in place.')

---
## Step 4 — Configure scene + camera subset
`CAMERAS` = the few cameras to process (fewer = faster). Set to `None` for all 12.

In [ ]:
SCENE   = 'Warehouse_016'
DATASET = 'Val'                      # Val | Test
CAMERAS = ['Camera', 'Camera_01', 'Camera_02', 'Camera_03']   # or None for all

os.chdir(REPO)
print(f'Scene   : {SCENE}  ({DATASET})')
print(f'Cameras : {CAMERAS if CAMERAS else "ALL"}')

---
## Step 5 — Download dataset from HuggingFace *(Colab only)*
Videos + calibration + ground_truth for the scene. Cached to Drive after first run.
Needs `HF_TOKEN` in Colab Secrets (🔑) and accepted dataset terms.

In [ ]:
if ON_COLAB:
    import shutil, getpass
    os.system('pip install -q huggingface_hub')
    from huggingface_hub import snapshot_download, login
    drive_data   = f'{DRIVE}/datasets/{DATASET}/{SCENE}'
    local_data   = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    drive_videos = f'{drive_data}/videos'
    videos_local = f'{local_data}/videos'
    os.makedirs(drive_data, exist_ok=True); os.makedirs(local_data, exist_ok=True)
    if os.path.exists(drive_videos) and os.listdir(drive_videos):
        print('[CACHE HIT] videos already on Drive.')
    else:
        hf_token = None
        try:
            from google.colab import userdata; hf_token = userdata.get('HF_TOKEN')
        except Exception: pass
        if not hf_token: hf_token = getpass.getpass('Paste HuggingFace token: ')
        login(token=hf_token)
        hf_split = DATASET.lower()
        print(f'Downloading {SCENE} ({DATASET})...', flush=True)
        snapshot_download(repo_id='nvidia/PhysicalAI-SmartSpaces', repo_type='dataset',
            local_dir='/content/hf_tmp',
            allow_patterns=[f'MTMC_Tracking_2025/{hf_split}/{SCENE}/videos/**',
                            f'MTMC_Tracking_2025/{hf_split}/{SCENE}/calibration.json',
                            f'MTMC_Tracking_2025/{hf_split}/{SCENE}/ground_truth.json'])
        hf_src = f'/content/hf_tmp/MTMC_Tracking_2025/{hf_split}/{SCENE}'
        if not os.path.exists(drive_videos): shutil.copytree(f'{hf_src}/videos', drive_videos)
        for fn in ['calibration.json','ground_truth.json']:
            if os.path.exists(f'{hf_src}/{fn}'): shutil.copy(f'{hf_src}/{fn}', f'{drive_data}/{fn}')
        shutil.rmtree('/content/hf_tmp', ignore_errors=True)
    for fn in ['calibration.json','ground_truth.json']:
        s,d = f'{drive_data}/{fn}', f'{local_data}/{fn}'
        if os.path.exists(s) and not os.path.exists(d): shutil.copy(s,d)
    if os.path.islink(videos_local): os.unlink(videos_local)
    os.makedirs(videos_local, exist_ok=True)
    os.makedirs(f'{local_data}/depth_map', exist_ok=True)
    cams_avail = sorted(os.path.splitext(f)[0] for f in os.listdir(drive_videos) if f.endswith('.mp4'))
    print(f'✓ Dataset ready — {len(cams_avail)} cameras available: {cams_avail}')
else:
    print('Local: using existing AIC25_Track1/ data.')

---
## Link outputs to Drive (persist across disconnects)

In [ ]:
if ON_COLAB:
    import shutil as _shutil
    for folder in ['Detection','EmbedFeature','Tracking']:
        drive_folder = f'{DRIVE}/outputs/{folder}'; repo_folder = f'{REPO}/{folder}'
        os.makedirs(drive_folder, exist_ok=True)
        if os.path.islink(repo_folder): pass
        elif os.path.isdir(repo_folder): _shutil.move(repo_folder, drive_folder); os.symlink(drive_folder, repo_folder)
        else: os.symlink(drive_folder, repo_folder)
        print(f'{folder}/ → Drive')
else:
    print('Local: outputs saved directly in repo.')

---
## Generate single-camera JSONs — fallback detector, NO training
Order matters: **detection → embedding → tracking → fix**, and frames are kept until tracking is done (embedding *and* tracking both read the image frames). Frames are deleted only at the end.

In [ ]:
import os, shutil, cv2, subprocess
os.chdir(REPO)

_dv = f'{DRIVE}/datasets/{DATASET}/{SCENE}/videos' if DRIVE else None
_lv = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos'
cam_source = _dv if (_dv and os.path.exists(_dv)) else _lv
_items   = os.listdir(cam_source)
_mp4s    = sorted(os.path.splitext(f)[0] for f in _items if f.endswith('.mp4'))
_dirs    = sorted(c for c in _items if os.path.isdir(f'{cam_source}/{c}') and 'map' not in c)
all_cams = _mp4s if _mp4s else _dirs
cams = [c for c in CAMERAS if c in all_cams] if CAMERAS else all_cams
missing = [c for c in (CAMERAS or []) if c not in all_cams]
if missing: print('WARN — not in dataset, skipping:', missing)
print('Processing cameras:', cams)

if os.path.exists(f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'):
    ckpt='BoT-SORT/ai_city_ckpt.pth.tar'; exp_file='BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py'
    print('Detector: AIC25-trained')
else:
    ckpt='BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; exp_file='BoT-SORT/yolox/exps/example/mot/yolox_x_mix_det.py'
    print('Detector: ByteTrack FALLBACK (no training)')

def extract_frames(cam):
    fd = f'{_lv}/{cam}/Frame'
    if os.path.exists(fd) and os.listdir(fd):
        print(f'  [B] {cam}: frames exist ({len(os.listdir(fd))})'); return fd
    mp4 = f'{cam_source}/{cam}.mp4'
    if not os.path.exists(mp4):
        d = f'{cam_source}/{cam}'
        mp4 = next((f'{d}/{f}' for f in os.listdir(d) if f.endswith('.mp4')), None) if os.path.isdir(d) else None
    if not mp4: print(f'  [ERR] no mp4 for {cam}'); return None
    os.makedirs(fd, exist_ok=True)
    cap = cv2.VideoCapture(mp4); n=1; ok,frm = cap.read()
    while ok:
        cv2.imwrite(f'{fd}/{str(n).zfill(6)}.jpg', frm); ok,frm = cap.read()
        if n % 2000 == 0: print(f'    {cam}: {n} frames', flush=True)
        n += 1
    cap.release(); print(f'  [B] {cam}: extracted {n-1} frames'); return fd

def run(cmd):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode != 0: print('\n'.join(r.stdout.strip().splitlines()[-40:]))
    return r.returncode

# PHASE 1 — frames + detection
for cam in cams:
    print(f'\n=== [B+C] {cam} ===')
    if extract_frames(cam) is None: continue
    if os.path.exists(f'{REPO}/Detection/{SCENE}/{cam}.txt'):
        print('  [C] detection exists — skip'); continue
    rc = run(f'{PY} BoT-SORT/tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} '
             f'--camera {cam} -f {exp_file} -c {ckpt} ./')
    print('  [C] detection', 'OK' if rc==0 else f'FAIL {rc}')

# Safety: embedding is scene-wide — ensure frames exist for EVERY camera with a detection file
os.chdir(REPO)
det_dir = f'{REPO}/Detection/{SCENE}'
det_cams = [os.path.splitext(f)[0] for f in os.listdir(det_dir) if f.endswith('.txt')] if os.path.isdir(det_dir) else []
for cam in det_cams:
    fd = f'{_lv}/{cam}/Frame'
    if not (os.path.exists(fd) and os.listdir(fd)):
        print(f'  embedding needs frames for {cam} (has detection) — extracting'); extract_frames(cam)

# PHASE 2 — embeddings (OSNet, scene-wide)
print('\n=== [D] embeddings (OSNet) ===')
os.chdir(f'{REPO}/deep-person-reid')
rc = os.system(f'{PY} torchreid/aic25_extract.py -s {SCENE} --dataset {DATASET} ../')
print('  [D]', 'OK' if rc==0 else f'FAIL {rc}')
os.chdir(REPO)

# PHASE 3 — tracking + fix (frames still present)
for cam in cams:
    print(f'\n=== [E+F] {cam} ===')
    rc = run(f'{PY} BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('  [E] tracking', 'OK' if rc==0 else f'FAIL {rc}')
    rc = run(f'{PY} BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('  [F] fix', 'OK' if rc==0 else f'FAIL {rc}')

# PHASE 4 — free local frames
for cam in cams:
    shutil.rmtree(f'{_lv}/{cam}', ignore_errors=True)
print(f'\n[DONE] JSONs at Tracking/Singlecamera/{SCENE}/<camera>/{{<camera>.json, fixed_<camera>.json}}')

---
## Tier-1 — Tracklet Repair Ablation
Runs the 4-way ablation (`baseline` / `interpolation_only` / `merge_only` / `full_repair`) on each camera's **raw** tracking JSON and prints a combined table. Per-camera `ablation.md` / `ablation.json` are saved under `tracklet_repair/results/`.

In [ ]:
import os, json, subprocess
os.chdir(REPO)
VARIANTS = ['baseline','interpolation_only','merge_only','full_repair']
METRICS  = ['total_detections','num_tracklets','mean_tracklet_length',
            'num_tracklets_with_gaps','total_internal_gaps','interpolated_detections','merged_tracklets']
summary = {}
for cam in cams:
    raw = f'Tracking/Singlecamera/{SCENE}/{cam}/{cam}.json'
    if not os.path.exists(raw): print(f'[skip] {cam}: no {raw}'); continue
    outdir = f'tracklet_repair/results/ablation_{SCENE}_{cam}'
    subprocess.run(f'{PY} -m tracklet_repair.src.evaluation.run_ablation --input-json {raw} '
                   f'--output-dir {outdir} --max-gap 5 --max-merge-gap 5 --max-center-distance 80 '
                   f'--max-size-ratio 1.5 --short-threshold 10', shell=True)
    try:
        abl = json.load(open(f'{outdir}/ablation.json'))
        summary[cam] = {v: abl['variants'][v]['statistics'] for v in VARIANTS}
    except Exception as e:
        print(f'[warn] {cam}: could not read ablation.json ({e})')

print(f'\n{"="*70}\nTIER-1 SUMMARY — baseline vs full_repair (per camera)\n{"="*70}')
print(f'{"camera":<12}{"metric":<26}{"baseline":>10}{"full_repair":>13}')
print('-'*61)
for cam, vd in summary.items():
    for m in METRICS:
        b = vd['baseline'].get(m); f = vd['full_repair'].get(m)
        bs = f'{b:.2f}' if isinstance(b,float) else str(b)
        fs = f'{f:.2f}' if isinstance(f,float) else str(f)
        print(f'{cam:<12}{m:<26}{bs:>10}{fs:>13}')
    print()
print('Full 4-way tables: tracklet_repair/results/ablation_%s_<camera>/ablation.md' % SCENE)